In [ ]:
from option_chain_downloader import OptionChainDownloader
from option_finder import *
from option_data_plotter import *

In [ ]:
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/all_options.log')

In [ ]:
chain_dir = 'chain'
quotes_dir = 'quotes'
data_dir = 'data'
cookie_file = 'cookie.txt'
ocd = OptionChainDownloader(chain_dir, quotes_dir, cookie_file, logger, strikes='ALL')
self = OptionFinder(logger, chain_dir=chain_dir, report_dir=data_dir)
print('OptionFinder chain_dir:', self.chain_dir)

In [ ]:
symlist  = ['QQQ', 'SPY', 'DIA', 'GLD', 'TLT', 'IBIT']
symlist += ['TSM', 'CRCL', 'NVDA', 'AAPL', 'GOOGL', 'MSFT']
symlist += ['TSLA', 'PLTR', 'META', 'AMZN', 'AVGO', 'SMH', 'ETHA']

## Refresh data here

In [ ]:
_t0 = time.time()
_n = ocd.download_option_chain(symlist, batch_size=5, rps=5)
print(_n, 'file downloads requested in', int(time.time() - _t0), 'seconds')

In [ ]:
self.get_quote_df(symlist)
_t0 = time.time()
_df = self.build_option_df(symlist)
_t1 = time.time()
print(f'build_option_df {_t1 - _t0:.1f} seconds')
dfcp = self.concat_put_call_options(_df)
dfcp = bucketize_dte(add_moneyness_columns(dfcp))
_t2 = time.time()
print(f'concat_put_call_options {_t2 - _t1:.1f} seconds')
px.bar(check_data_age(_df), y=['load_age', 'quote_age'], barmode='group', title=f"Data Ages", width=800, height=300).show()
print(f'px.bar {time.time() - _t2:.1f} seconds')
dfcp.loc[:, ['dte', 'expDt']].groupby('dte').first().head(24).tail(20).T

In [ ]:
get_closest_value_in_column(dfcp, 'dte', 45)

### ImpVola Overview

In [ ]:
plot_iv(dfcp)

In [ ]:
_symbol = 'QQQ'
_strike = 511
_strike = get_closest_value_in_column(dfcp[dfcp.symbol==_symbol], 'strike', _strike)
px.scatter(get_theta_curves(dfcp, _symbol, _strike), title=f'Theta curves of {_symbol} strike {_strike}', height=500)

### Days to zero: smaller theta gives longer dtz

In [ ]:
def compute_dtz(dfcp, symbol, opt_type, dte, strike, debug=False):
    _df = dfcp[(dfcp.symbol == symbol) & (dfcp.dte==dte) & (dfcp.type==opt_type) & (dfcp.strike==strike)]
    spot_price = _df.lastPrice.iloc[0]
    premium = _df.mid.iloc[0]
    theta_curve = get_theta_curves(dfcp[dfcp.dte <= dte], symbol, strike)[opt_type]
    if theta_curve.shape[0] == 1:
        theta = theta_curve.iloc[0]
        dtz = premium/(0 - theta)
        dth = dtz/2
        if debug:
            print(f'find_zero: single dte: premimu {premium}, theta {theta}, dtz {dtz}, dth {dth}')
        if dtz <= dte:
            return dth, dtz, 0
        resid = premium *(dtz/dte - 1)
        if dth <= dte:
            return dth, dte, resid
        return np.nan, dte, resid
    df_thc = theta_curve[theta_curve.index <= dte].T.reset_index().dropna().rename(columns={opt_type: 'theta'})
    df_thc['diff_dte'] = df_thc['dte'].diff()
    df_thc['avg_theta'] = df_thc['theta'].rolling(window=2).mean()
    df_thc['theta_decay'] = df_thc['diff_dte'] * df_thc['avg_theta']
    total_decay = df_thc['theta_decay'].sum()
    if debug:
        print('Spot price:', spot_price, 'Premium:', premium, 'total decay:', total_decay)
    df_thc['decay_cumsum']  = df_thc['theta_decay'][::-1].cumsum()[::-1]
    if debug:
        df_thc['dte_cumsum'] = df_thc['diff_dte'][::-1].cumsum()[::-1]
    df_thc['half_resid'] = premium/2 + df_thc['decay_cumsum']
    df_thc['resid'] = premium + df_thc['decay_cumsum']
    def find_zero(resid_col):
        # resid_col is always ascending
        if df_thc[resid_col].dropna().iloc[0] >= 0:
                # all positive
                if debug:
                    print(f'find_zero: {resid_col} all positive', df_thc[resid_col])
                return dte
        if df_thc.iloc[-1][resid_col] <= 0:
            # all negative
            if debug:
                print(f'find_zero: {resid_col} all negative', list(df_thc[resid_col]))
            row = df_thc.iloc[-1]
            adj = row['diff_dte'] * row[resid_col]/row['theta_decay']
            return dte - (row['dte'] - row['diff_dte'] - adj)
        else:
            neg_filter = df_thc[resid_col] <= 0
            neg_idx = df_thc[neg_filter]['dte'].idxmax()
            pos_filter = df_thc[resid_col] >= 0
            pos_idx = df_thc[pos_filter]['dte'].idxmin()
            if debug:
                print(f'find_zero: {resid_col}: choice between neg_idx.max {neg_idx} and pos_idx.min {pos_idx}')
            neg_row = df_thc.loc[neg_idx]
            pos_row = df_thc.loc[pos_idx]
            neg_ratio = np.abs(neg_row[resid_col]/neg_row['theta_decay'])
            pos_ratio = np.abs(pos_row[resid_col]/pos_row['theta_decay'])
            row = df_thc.loc[neg_idx] if neg_ratio < pos_ratio else df_thc.loc[pos_idx]
            adj = row['diff_dte'] * row[resid_col]/row['theta_decay']
            if debug:
                print(f'find_zero: {resid_col}: resid ratio: {neg_ratio} vs {pos_ratio}, {row[resid_col]}, adj: {adj}')
                print(f'-- row: {row.to_dict()}')
            if row[resid_col] > 0:
                # adj is negative, do extrapolation
                return dte - (row['dte'] + row['diff_dte'] + adj)
            else:
                # do interpolation
                return dte - (row['dte'] - adj)
    dth = find_zero('half_resid')
    if premium + total_decay > 0:
        if debug:
            print('dtz > dte', 'dth:', dth, 'resid:', (premium + total_decay)/premium)
        else:
            return dth, dte, (premium + total_decay)/premium
    else:
        dtz = find_zero('resid')
        if debug:
            print('dth:', dth, 'dtz:', dtz)
        else:
            return dth, dtz, 0
    return df_thc

In [ ]:
_symbol = 'DIA'
_strike = 492
px.scatter(get_theta_curves(dfcp[(dfcp.symbol==_symbol) & (dfcp.strike == _strike) & (dfcp.type=='P')], _symbol, _strike))

In [ ]:
#(dfcp.symbol=='DIA') & (dfcp.dte<=349) & (dfcp.strike==492)
_symbol = 'DIA'
_dte = 349
_strike = 492
_dte = get_closest_value_in_column(dfcp[dfcp.symbol==_symbol], 'dte', _dte)
_strike = get_closest_value_in_column(dfcp[(dfcp.symbol==_symbol) & (dfcp.dte == _dte)], 'strike', _strike)
# Alternative. This could end up with a different result
#_strike = get_closest_value_in_column(dfcp[dfcp.symbol==_symbol], 'strike', _strike)
#_dte = get_closest_value_in_column(dfcp[(dfcp.symbol==_symbol) & (dfcp.strike == _strike)], 'dte', _dte)
print('dte:', _dte, 'strike:', _strike)
_df = compute_dtz(dfcp, _symbol, 'P', _dte, _strike, debug=True)
_df

In [ ]:
def compute_all_dtz_for_symbol(dfcp, symbol, opt_type):
    dfp = dfcp[(dfcp.symbol==symbol) & (dfcp.type==opt_type) & (dfcp.dte <= 365)]
    dte_list = dfp.dte.unique()
    res = []
    for dte in dte_list:
        if dte == 0:
            continue
        dte = dte.item()
        strike_list = dfp[dfp.dte==dte].strike.unique()
        for strike in strike_list:
            strike = strike.item()
            _filter = (dfp.dte==dte) & (dfp.strike==strike) & (dfp.type=='P')
            if dfp[_filter]['moneyness'].iloc[0] >= 1.0 or dfp[_filter]['pctProfit'].iloc[0] <= 0.5:
                continue
            dth, dtz, resid = compute_dtz(dfp, symbol, 'P', dte, strike)
            res.append({'symbol': symbol, 'dte': dte, 'strike': strike, 'dth': dth, 'dtz': dtz, 'resid': resid})
    return pd.DataFrame(res)

In [ ]:
dfcp[(dfcp.symbol=='QQQ') & (dfcp.strike==620) & (dfcp.type=='P')].drop(columns=dfcp.columns[-4:])

In [ ]:
lodf = [compute_all_dtz_for_symbol(dfcp, symbol, 'P') for symbol in ['QQQ', 'SPY', 'GLD', 'DIA']]

In [ ]:
dfp = pd.concat(lodf)
dfp['dthr'] = dfp.dth/dfp.dte
dfp['dtzr'] = dfp.dtz/dfp.dte

In [ ]:
dfcp[(dfcp.type=='P') & (dfcp.symbol=='DIA') & (dfcp.dte<=349) & (dfcp.strike==492)].drop(columns=dfcp.columns[-5:])

In [ ]:
dfp.sort_values(by='dthr').head(60)

In [ ]:
_symbol = 'QQQ'
_strike = 625
px.scatter(get_theta_curves(dfcp[(dfcp.symbol==_symbol) & (dfcp.strike == _strike)], _symbol, _strike))

In [ ]:
_dte = get_closest_value_in_column(dfcp, 'dte', 180)
_filter = (dfcp.type=='C') & (dfcp.dte==_dte) & dfcp.cluster.str.contains(r'^otm_short') & (dfcp.symbol != 'TLT')
px.scatter(dfcp[_filter], x='moneyness', y='dtz', color='symbol', title=f'Days to Zero - OTM short calls {_dte} DTE')

In [ ]:
_dte = get_closest_value(dfcp.loc[:, ['dte']], 'dte', 30).iloc[0, 0]
_filter = (dfcp.type=='P') & (dfcp.dte==_dte) & dfcp.cluster.str.contains(r'^otm_short') & (dfcp.symbol == 'QQQ')
px.scatter(dfcp[_filter], x='strike', y='dtz', color='Delta', title=f'Days to Zero - OTM short puts {_dte} DTE')

In [ ]:
plot_theta_of_symbol(dfcp[~dfcp.cluster.str.contains('deep')], 'QQQ')

In [ ]:
plot_theta_of_symbol(dfcp[~dfcp.cluster.str.contains('deep')], 'GOOGL')

### Holy Grail of Puts

In [ ]:
_filter = (dfcp.type=='P') & (dfcp.pctProfit >= 1.0) & (dfcp.strike <= dfcp.lastPrice)
dfp = dfcp[_filter].drop(columns=['type', 'dte_cluster', 'leverage', 'cluster', 'overpaid', 'Rho'])

In [ ]:
_df = dfp[dfp.symbol=='GOOGL'].groupby('strike')['dte'].min().reset_index()
px.scatter(_df, x='strike', y='dte')

In [ ]:
_dfp = dfp[(dfp.pctSpread <= 4) & (dfp.dtz <= 14) & (dfp.mid >= 1) & (dfp.OpenInterest >= 100)].sort_values(by='dtzr')
_dfp.head(30)

In [ ]:
dfp[(dfp.symbol == 'TSLA') & (dfp.strike <= 385) & (dfp.dtz <= 30)].sort_values(by='dtzr').head(60)

In [ ]:
dfl = plot_leverage_overpaid(dfcp[(dfcp.dte >= 90) & (dfcp.dte <= 360)], delta_lb=0.5, overpaid_ub=0.05, price_lb=5, spread_ub=5, leverage_lb=2, openinterest_lb=100)
dfl = dfl.drop(columns=['pctProfit', 'type', 'Rho', 'cluster', 'dte_cluster'])

In [ ]:
_filter = (dfl.index.get_level_values(0) == 'GOOGL') & (dfl.dtzr >= 10)
dfl[_filter].sort_values(by='leverage', ascending=False).head(60)

### Put Debit Spread

In [ ]:
def calc_pds_debit(df):
    strike0 = df.strike.iloc[0]
    spreads = list(df.strike.iloc[[1, -1]] - strike0)
    return int((df.mid.iloc[0]*2 + df.mid.iloc[1] - df.mid.iloc[-1])*100)/100, strike0, spreads

def get_final_dfpds(dfpds):
    pds_list = []
    for idx in dfpds.index:
        _df = put_debit_spread(dfpds, *idx, dfcp)
        debit, strike0, spreads = calc_pds_debit(_df)
        pds_list.append({'symbol': idx[0], 'dte': idx[1], 'debit': debit, 'strike0': strike0, 'spread1': spreads[0], 'spread2': spreads[1]})
    return dfpds.join(pd.DataFrame(pds_list, index=dfpds.index))

In [ ]:
dfpds = select_pds_deltas(dfcp, 35, 45)
dfpds

### 30 days dte put options

In [ ]:
dfp30 = dfcp[(dfcp.type=='P') & (dfcp.dte >= 14) & (dfcp.dte <= 49) & (dfcp.Delta >= -0.45) & (dfcp.pctSpread <= 10)]
dfp30 = dfp30.drop(columns=['type', 'dte_cluster', 'leverage', 'cluster', 'overpaid']).sort_values(by='pctProfit', ascending=False)
dfp30.head(60)

In [ ]:
px.scatter(dfp30[(dfp30.symbol=='QQQ') & (dfp30.strike <= 0.95*dfp30.lastPrice)], x='Delta', y='mid', color='dte')

In [ ]:
px.scatter(dfp30[dfp30.pctProfit >= 1].head(60), x='moneyness', y='pctProfit', color='symbol')

### 0 DTE PUT

## Total open interests and volumes for all dte and strikes

In [ ]:
_df = dfcp.loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False)

### It may be of interest to look at OpenInterests and Volumes for 8-weeks and 1-year DTE clusters

In [ ]:
for _dte_cluster in ['8wk', '1yr']:
    _df = dfcp[dfcp.dte_cluster==_dte_cluster].loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
    plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False, log_y_threshold=500, horizontal_spacing=0.03)

### Ratios of volume/openinterest

In [ ]:
dfvo = aggregate_metrics_by_moneyness_and_dte_clusters(dfcp, ['Volume', 'OpenInterest'], method='sum', oi_lb=None, bid_lb=0)
dfvo['Volume_OpenInterest_ratio'] = dfvo.Volume/dfvo.OpenInterest
dfvo = dfvo.drop(columns=['Volume', 'OpenInterest'])

In [ ]:
_dte_cluster = '1wk'
for _type in ['C', 'P']:
    _df = dfvo[(dfvo.dte_cluster==_dte_cluster) & (dfvo.type==_type)].drop(columns=['dte_cluster', 'type'])
    _metric = _df.columns[-1]
    _metric2 = f'{_type} option {_dte_cluster} {_metric}'
    plot_metrics_in_one_row(_df.rename(columns={_metric: _metric2}), ['symbol', 'cluster'], [_metric2], shared_y=False, log_y_threshold=500, horizontal_spacing=0.02)

In [ ]:
plot_metrics_by_moneyness_and_dte_clusters(dfvo[(dfvo.type=='C') & (dfvo.dte_cluster != '1wk')], height=240, vertical_spacing=0.03, shared_yaxes=True)

In [ ]:
_df = calc_overall_put_call_ratios(dfcp)
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_df = calc_overall_put_call_ratios(dfcp[dfcp.cluster=='atm'])
_df = _df.rename(columns=dict([(c, 'atm '+c) for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_dte_lb = 90
_dte_ub = 180
_df = calc_overall_put_call_ratios(dfcp[(dfcp.cluster=='atm') & (dfcp.dte >= _dte_lb) & (dfcp.dte <= _dte_ub)])
_df = _df.rename(columns=dict([(c, f'atm dte {_dte_lb} to {_dte_ub} {c}') for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:], shared_y=False)

In [ ]:
dte_filter = ~ dfcp.dte_cluster.str.contains('^(?:1w|>1y)')
_df = calc_cluster_put_call_ratios(dfcp[dte_filter])#[~ (dfcp.cluster.str.contains('deep')
plot_metrics_by_moneyness_and_dte_clusters(_df, height=230, vertical_spacing=0.03, shared_yaxes=False)

Which metrics can be clustered by moneyness/dte? How to compare options between symbols?
pctProfit, pctSpread, theta, overpaid, leverage

### Average pctSpread of put options by moneyness and DTE clusters

In [ ]:
_df = aggregate_metrics_by_moneyness_and_dte_clusters(dfcp, 'pctSpread', method='mean', oi_lb=100, bid_lb=0)
_filter = (_df.type == 'P') & (~ _df.dte_cluster.str.contains(r'^(?:1wk|>1yr)')) & (~ _df.cluster.str.contains(r'^(?:deep)'))
plot_metrics_by_moneyness_and_dte_clusters(_df[_filter], height=230, vertical_spacing=0.03, shared_yaxes=False)

In [ ]:
_symbol = 'QQQ'
_df = dfcp[dfcp.symbol==_symbol]
oi_lb = _df[_df.OpenInterest>0].OpenInterest.quantile(0.1)
print('OpenInterest lb:', oi_lb, 'Spot price:', _df.lastPrice.values[0])
px.scatter(_df[(_df.OpenInterest >= oi_lb) & (_df.Bid >= 10)], x='OpenInterest', y='pctSpread', color='type')

In [ ]:
dfcp.dte.unique()

In [ ]:
_dte = 183
_metric = 'pctTheta'#'OpenInterest'
dfcp['pctTheta'] = dfcp.Theta/dfcp.mid*100
_df = dfcp[(dfcp.symbol==_symbol) & (dfcp.dte==_dte)]
px.scatter(_df[(_df.moneyness >= 0.9) & (_df.moneyness <= 0.99)], x='strike', y=_metric, color='type', title=f'{_symbol} {_metric} dte: {_dte}')

In [ ]:
_p = _df[_df.moneyness <= 0.99].pivot(columns=['type'], index='strike', values=['mid'])
_p['p_c'] = _p.mid.P / _p.mid.C
_p = _p.reset_index()
_p.columns = ['strike', 'mid_C', 'mid_P', 'p_c']
#_p.reset_index().columns
px.scatter(_p, x='strike', y='p_c', title=f'{_symbol} put/call premium ratio')

In [ ]:
_symbol = 'QQQ'
_ = plot_put_call_ratios_by_moneyness(dfcp[(dfcp.symbol==_symbol) & (dfcp.dte >= 90)])

In [ ]:
_df = plot_option_details_by(dfcp, 'SPY', 'Theta', 'dte', offset=10, delta_lb=0.25, delta_ub=0.85, nr=4, nc=2)

### The End